In [1]:
pip install pandas numpy scikit-learn matplotlib seaborn


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas, numpy, sklearn
import matplotlib.pyplot as plt
import seaborn as sns       


loading my datasets

In [5]:

import zipfile

zip_path = r"C:\Users\Admin\Desktop\DATA SCIENCE\semester 3\Text mining and search\text-clustering\product+classification+and+clustering (4).zip"

with zipfile.ZipFile(zip_path, "r") as z:
    print("Files in zip:", z.namelist())
    with z.open("pricerunner_aggregate.csv") as f:
        df = pandas.read_csv(f)

print(df.shape)
df.head()



Files in zip: ['pricerunner_aggregate.csv']
(35311, 7)


,Product ID,Product Title,Merchant ID,Cluster ID,Cluster Label,Category ID,Category Label
0,1,apple iphone 8 plus 64gb silver,1,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
1,2,apple iphone 8 plus 64 gb spacegrau,2,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
2,3,apple mq8n2b/a iphone 8 plus 64gb 5.5 12mp sim...,3,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
3,4,apple iphone 8 plus 64gb space grey,4,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
4,5,apple iphone 8 plus gold 5.5 64gb 4g unlocked ...,5,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones


Clean columns + keep what we need

In [ ]:
# Clean column names (remove leading/trailing spaces)
df.columns = [c.strip() for c in df.columns]

# Keep only the columns needed for clustering + evaluation
df = df[["Product Title", "Category Label"]].copy()

# Basic cleaning
df["Product Title"] = df["Product Title"].astype(str)

# Drop missing and duplicates (optional but helpful)
df = df.dropna(subset=["Product Title", "Category Label"])
df = df.drop_duplicates(subset=["Product Title"])

print(df.shape)
df.head()


(30993, 2)


,Product Title,Category Label
0,apple iphone 8 plus 64gb silver,Mobile Phones
1,apple iphone 8 plus 64 gb spacegrau,Mobile Phones
2,apple mq8n2b/a iphone 8 plus 64gb 5.5 12mp sim...,Mobile Phones
3,apple iphone 8 plus 64gb space grey,Mobile Phones
4,apple iphone 8 plus gold 5.5 64gb 4g unlocked ...,Mobile Phones


Text preprocessing

We’ll do light cleaning:
lowercase, remove weird characters, keep words/numbers (useful for products like “64gb”, “i7”, etc.)

In [ ]:
import re

def clean_text(s: str) -> str:
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)     # keep letters, numbers, spaces
    s = re.sub(r"\s+", " ", s).strip()     # remove extra spaces
    return s

df["clean_title"] = df["Product Title"].apply(clean_text)
df[["Product Title", "clean_title"]].head(10)


,Product Title,clean_title
0,apple iphone 8 plus 64gb silver,apple iphone 8 plus 64gb silver
1,apple iphone 8 plus 64 gb spacegrau,apple iphone 8 plus 64 gb spacegrau
2,apple mq8n2b/a iphone 8 plus 64gb 5.5 12mp sim...,apple mq8n2b a iphone 8 plus 64gb 5 5 12mp sim...
3,apple iphone 8 plus 64gb space grey,apple iphone 8 plus 64gb space grey
4,apple iphone 8 plus gold 5.5 64gb 4g unlocked ...,apple iphone 8 plus gold 5 5 64gb 4g unlocked ...
6,apple iphone 8 plus 64 gb space grey,apple iphone 8 plus 64 gb space grey
10,apple iphone 8 plus 5.5 single sim 4g 64gb silver,apple iphone 8 plus 5 5 single sim 4g 64gb silver
11,sim free iphone 8 plus 64gb by apple space grey,sim free iphone 8 plus 64gb by apple space grey
12,apple iphone 8 plus 64gb gold smartphone,apple iphone 8 plus 64gb gold smartphone
13,apple iphone 8 plus 5.5 single sim 4g 64gb grey,apple iphone 8 plus 5 5 single sim 4g 64gb grey


Convert text to numbers using TF-IDF
Meaning of TF-IDF

TF = Term Frequency (how often a word appears in a title)
IDF = Inverse Document Frequency (reduces weight of very common words)
Result: each title becomes a numeric vector

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",      # removes common English words like "the", "and"
    ngram_range=(1, 2),        # use single words + pairs like "iphone 8"
    min_df=3,                  # ignore very rare tokens
    max_df=0.90                # ignore extremely common tokens
)

X = tfidf.fit_transform(df["clean_title"])
print("TF-IDF matrix shape:", X.shape)


TF-IDF matrix shape: (30993, 17613)


Choosing number of clusters (K)

Search for the best K by printing the silhoutte scores


In [ ]:
scores = {}
for k in range(5, 21):
    km = KMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = km.fit_predict(X)
    scores[k] = silhouette_score(X, labels)
    sil = silhouette_score(X, cluster_labels)
print("Silhouette score:", sil)

df["cluster"] = cluster_labels
df.head()

scores

Silhouette score: 0.02370856604271429


{5: 0.013945804554865835,
 6: 0.012016591968349668,
 7: 0.016099965912281376,
 8: 0.02146670723529484,
 9: 0.021756316068141622,
 10: 0.02370856604271429,
 11: 0.026095613774127148,
 12: 0.025198120431190883,
 13: 0.02613311054253567,
 14: 0.025087896995222993,
 15: 0.026490458411094875,
 16: 0.026565687738482512,
 17: 0.027788517002261523,
 18: 0.027856224844932963,
 19: 0.027974972402380165,
 20: 0.030034591631461202}

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

K = 20

kmeans = KMeans(n_clusters=K, random_state=42, n_init="auto")
cluster_labels = kmeans.fit_predict(X)

sil = silhouette_score(X, cluster_labels)
print("Silhouette score:", sil)

df["cluster"] = cluster_labels
df.head() 

Silhouette score: 0.030034591631461202


,Product Title,Category Label,clean_title,cluster
0,apple iphone 8 plus 64gb silver,Mobile Phones,apple iphone 8 plus 64gb silver,8
1,apple iphone 8 plus 64 gb spacegrau,Mobile Phones,apple iphone 8 plus 64 gb spacegrau,8
2,apple mq8n2b/a iphone 8 plus 64gb 5.5 12mp sim...,Mobile Phones,apple mq8n2b a iphone 8 plus 64gb 5 5 12mp sim...,8
3,apple iphone 8 plus 64gb space grey,Mobile Phones,apple iphone 8 plus 64gb space grey,8
4,apple iphone 8 plus gold 5.5 64gb 4g unlocked ...,Mobile Phones,apple iphone 8 plus gold 5 5 64gb 4g unlocked ...,8


Evaluate clustering using the true labels (Category Label)

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

le = LabelEncoder()
y_true = le.fit_transform(df["Category Label"])

ari = adjusted_rand_score(y_true, df["cluster"])
nmi = normalized_mutual_info_score(y_true, df["cluster"])

print("ARI:", ari)
print("NMI:", nmi)
print("Categories:", list(le.classes_))


ARI: 0.3024897852142445
NMI: 0.6041745220238777
Categories: ['CPUs', 'Digital Cameras', 'Dishwashers', 'Freezers', 'Fridge Freezers', 'Fridges', 'Microwaves', 'Mobile Phones', 'TVs', 'Washing Machines']


how clusters map to real categories

In [ ]:
ct = pd.crosstab(df["cluster"], df["Category Label"])
print(ct.head())

# Show dominant label per cluster
dominant = ct.idxmax(axis=1)
dominant_counts = ct.max(axis=1)

summary = pd.DataFrame({
    "dominant_category": dominant,
    "dominant_count": dominant_counts,
    "cluster_size": ct.sum(axis=1)
}).sort_index()

print(summary.head())


Category Label  CPUs  Digital Cameras  Dishwashers  Freezers  Fridge Freezers  \
cluster                                                                         
0                  0             1161            0         0                0   
1                  0                0            2       713               75   
2                  0                1            0         0                0   
3                  0                0           35        69               63   
4               1033                1            0         0                0   

Category Label  Fridges  Microwaves  Mobile Phones   TVs  Washing Machines  
cluster                                                                     
0                     0           5            152    33                 0  
1                  1207           0              1     0                 3  
2                     0           0              0  2333                 0  
3                    96          12            

Top keywords per cluster

In [ ]:
import numpy as np

terms = np.array(tfidf.get_feature_names_out())

order_centroids = kmeans.cluster_centers_.argsort(axis=1)[:, ::-1]

top_n = 12
for i in range(K):
    top_terms = terms[order_centroids[i, :top_n]]
    print(f"\nCluster {i} top terms: {', '.join(top_terms)}")



Cluster 0 top terms: camera, sony, digital, digital camera, lumix, panasonic, panasonic lumix, dmc, compact, sony xperia, black, xperia

Cluster 1 top terms: fridge, white, larder, larder fridge, freezer, integrated, counter, fridge white, freezer white, undercounter, tall, wide

Cluster 2 top terms: tv, hd, smart, led, 4k, hdr, led tv, ultra hd, ultra, smart tv, 4k ultra, inch

Cluster 3 top terms: zanussi, freezer, fridge, white, freezer white, fridge freezer, integrated, skala, skala bis, eek skala, bis, dishwasher

Cluster 4 top terms: core, intel, processor, intel core, ghz, ghz processor, core core, cache, i5, quad, quad core, retail

Cluster 5 top terms: machine, washing machine, washing, machine white, white, spin, 1400rpm, spin washing, 1400, 8kg, 1400 spin, 9kg

Cluster 6 top terms: microwave, oven, microwave oven, steel, built, built microwave, stainless, stainless steel, grill, solo, solo microwave, black

Cluster 7 top terms: fridge freezer, freezer, fridge, smeg, america